In [230]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import * 

### Introduction

This project uses two Spotify related datasets: `music_tracks.csv` and `artists.csv`. 

In [231]:
artists = pd.read_csv("data/artists.csv")
music_tracks = pd.read_csv("data/music_tracks.csv")

The `music_tracks.csv` dataset contains information for 114,000 tracks across 114 genres. Each row represents a single track, and includes both metadata and Spotify-generated audio features. These features include variables such as danceability, energy, acousticness, valence, tempo, explicit content, and popularity score. The `artists.csv` dataset contains artist-level information, including artist popularity, follower counts, and genre tags.

In this project, the primary focus is on the `music_tracks.csv` dataset because it contains track-level audio characteristics that may help explain why some songs become more popular than others.

The main research question I want to explore is: <br><br>
**Can we predict whether a Spotify song becomes popular based on its audio features and genre?** 

This question helps us better understand how audio characteristics and certain genres are associated with successful songs, providing insights into modern music trends and listener preferences. 

In [232]:
music_tracks.head()

,Unnamed: 0,track_id,artists,album_name,...,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,...,0.71,87.92,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),...,0.27,77.49,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,...,0.12,76.33,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,...,0.14,181.74,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,...,0.17,NaN,4,acoustic


In [233]:
music_tracks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   release_date      114000 non-null  object 
 8   explicit          114000 non-null  bool   
 9   danceability      114000 non-null  float64
 10  energy            114000 non-null  float64
 11  key               114000 non-null  int64  
 12  loudness          114000 non-null  float64
 13  mode              114000 non-null  int64  
 14  speechiness       114000 non-null  float64
 15  acousticness      114000 non-null  float64
 16  instrumentalness  11

In [234]:
artists.head()

,id,followers,genres,name,popularity
0,0DheY5irMjBUeLybbCUEZ2,0.0,[],Armid & Amir Zare Pashai feat. Sara Rouzbehani,0
1,0DlhY15l3wsrnlfGio2bjU,5.0,[],ปูนา ภาวิณี,0
2,0DmRESX2JknGPQyO15yxg7,0.0,[],Sadaa,0
3,0DmhnbHjm1qw6NCYPeZNgJ,0.0,[],Tra'gruda,0
4,0Dn11fWM7vHQ3rinvWEl4E,2.0,[],Ioannis Panoutsopoulos,0


In [235]:
artists.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1162095 entries, 0 to 1162094
Data columns (total 5 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   id          1162095 non-null  object 
 1   followers   1162084 non-null  float64
 2   genres      1162095 non-null  object 
 3   name        1162092 non-null  object 
 4   popularity  1162095 non-null  int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 44.3+ MB


In [236]:
print(music_tracks.shape)
print(artists.shape)

(114000, 22)
(1162095, 5)


In [237]:
music_tracks.describe()

,Unnamed: 0,popularity,duration_ms,danceability,...,liveness,valence,tempo,time_signature
count,114000.00,114000.00,1.14e+05,114000.00,...,114000.00,114000.00,91886.00,114000.00
mean,56999.50,33.24,2.28e+05,0.57,...,0.21,0.47,123.12,3.90
std,32909.11,22.31,1.07e+05,0.17,...,0.19,0.26,29.78,0.43
...,...,...,...,...,...,...,...,...,...
50%,56999.50,35.00,2.13e+05,0.58,...,0.13,0.46,123.00,4.00
75%,85499.25,50.00,2.62e+05,0.69,...,0.27,0.68,141.65,4.00
max,113999.00,100.00,5.24e+06,0.98,...,1.00,0.99,222.60,5.00


In [238]:
artists.describe()

,followers,popularity
count,1.16e+06,1.16e+06
mean,1.02e+04,8.80e+00
std,2.54e+05,1.36e+01
...,...,...
50%,5.70e+01,2.00e+00
75%,4.17e+02,1.30e+01
max,7.89e+07,1.00e+02


### Data Cleaning

In [239]:
# Check missing values
music_tracks.isnull().sum().sort_values(ascending=False)

tempo          22114
artists            1
album_name         1
               ...  
duration_ms        0
popularity         0
track_genre        0
Length: 22, dtype: int64

In [240]:
cleaned_tracks = music_tracks.drop(columns=["Unnamed: 0"])

In [241]:
cleaned_tracks.head()

,track_id,artists,album_name,track_name,...,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,...,0.71,87.92,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,...,0.27,77.49,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,...,0.12,76.33,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,...,0.14,181.74,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,...,0.17,NaN,4,acoustic


In [242]:
# Remove duplicates
print('Number of duplicate rows:', cleaned_tracks.duplicated().sum())
cleaned_tracks = cleaned_tracks.drop_duplicates()


Number of duplicate rows: 54


In [243]:
cleaned_tracks['release_date']

0               1974
1            1995-04
2               1973
             ...    
113997    1992-10-21
113998          1992
113999          1981
Name: release_date, Length: 113946, dtype: object

In [244]:
# Convert release date
cleaned_tracks['release_date'] = pd.to_datetime(
    cleaned_tracks['release_date'],
    errors = 'coerce'
)

# Extract year
cleaned_tracks['release_year'] = cleaned_tracks['release_date'].dt.year

In [245]:
# Inspect popularity distribution 
cleaned_tracks['popularity'].describe()

count    113946.00
mean         33.24
std          22.30
           ...    
50%          35.00
75%          50.00
max         100.00
Name: popularity, Length: 8, dtype: float64

In [246]:
# Finding threshold for popularity score

print((cleaned_tracks["popularity"] >= 50).value_counts(normalize=True))

print((cleaned_tracks["popularity"] >= 60).value_counts(normalize=True))

print((cleaned_tracks["popularity"] >= 70).value_counts(normalize=True))

print((cleaned_tracks["popularity"] >= 55).value_counts(normalize=True))

popularity
False    0.74
True     0.26
Name: proportion, dtype: float64
popularity
False    0.87
True     0.13
Name: proportion, dtype: float64
popularity
False    0.95
True     0.05
Name: proportion, dtype: float64
popularity
False    0.81
True     0.19
Name: proportion, dtype: float64


I selected 55 as the popularity threshold because it produced the most balanced class distribution out of all the tested thresholds while still representing moderately popluar songs. 

### Univariate Analysis

In [253]:
# Popularity Distribution
fig = cleaned_tracks['popularity'].hist(bins=20)
fig.show()

In [254]:
# Danceability Distribution
fig = cleaned_tracks['danceability'].hist(bins=20)
fig.show()

### Bivariate Analysis

In [255]:
# Danceability vs Popularity 
fig = cleaned_tracks.plot.scatter(
    x='danceability',
    y='popularity',
)

fig.show()

In [250]:
# Energy vs Popluarity
cleaned_tracks.plot.scatter(
    x='energy',
    y='popularity'
)


In [251]:
# Aggregate Statistics 

# To see which genres tend to have the highest average popularity
genre_popularity = (
    cleaned_tracks.groupby('track_genre')['popularity']
    .mean()
    .sort_values(ascending=False)
)
genre_popularity

track_genre
pop-film    59.28
k-pop       56.90
chill       53.65
            ...  
latin        8.31
romance      3.25
iranian      2.21
Name: popularity, Length: 114, dtype: float64

In [256]:
fig = genre_popularity.head(10).plot.bar()
fig.show()